In [ ]:
# 03 — Baselines: the bar the LSTM has to beat. Scores three simpler methods on
# the exact same test set the LSTM will use — seasonal persistence ("same as
# yesterday"), climatology ("the historical average for this hour"), and
# XGBoost (serious traditional ML). Saves their MAE/RMSE so 04 can compare
# against them.
# Input: feat_airquality.parquet
# Output: artifacts/baseline_metrics.csv

In [2]:
import sys; from pathlib import Path
ROOT = Path.cwd(); ROOT = ROOT if (ROOT/"common").exists() else ROOT.parent
sys.path.insert(0, str(ROOT/"modeling"))
import pandas as pd
from windowing import build_supervised
from splits import time_split                       # torch-free — do NOT import dataset here
from baselines import persistence, climatology, xgb_baseline, mae, rmse

df = pd.read_parquet(ROOT/"modeling"/"feat_airquality.parquet")
X, y, meta = build_supervised(df)                    # same split as 02_windows -> identical test set
tr, va, te = time_split(meta)

preds = {
    "seasonal_persistence": persistence(X[te]),
    "climatology":          climatology(X[tr], y[tr], meta.iloc[tr], X[te], meta.iloc[te]),
    "xgboost":              xgb_baseline(X[tr], y[tr], X[te]),
}
tbl = pd.DataFrame([{"model": k, "MAE": round(mae(p, y[te]),3), "RMSE": round(rmse(p, y[te]),3)}
                    for k, p in preds.items()])
print(tbl.to_string(index=False))
tbl.to_csv(ROOT/"modeling"/"artifacts"/"baseline_metrics.csv", index=False)
print("\nsaved -> modeling/artifacts/baseline_metrics.csv  (the bar for the LSTM)")

               model   MAE  RMSE
seasonal_persistence 2.879 5.019
         climatology 4.094 5.294
             xgboost 2.847 4.276

saved -> modeling/artifacts/baseline_metrics.csv  (the bar for the LSTM)
